Scarping for https://www.allfreecrochet.com/

In [1]:
import requests
import re
import time
import pandas as pd
from bs4 import BeautifulSoup

In [10]:
headers = {"User-Agent": "Mozilla/5.0"}

def scrape_product(product_link):
    row = {}
    r = requests.get(product_link, headers=headers)
    soup = BeautifulSoup(r.text, "html.parser")

    desc = soup.select_one("div.articleAttrSection")
    description = desc.get_text(strip=True)
    row["description"] = description

    sections = soup.select("div.articleAttrSection")

    for section in sections:
        label = section.find("span", class_="attrLabel")
        img = section.find("img")

        if label:
            key = label.get_text(strip=True)
            value = label.find_next("span").get_text(strip=True)
            row[key] = value
        elif img:
            row["skill_level"] = img.get("alt", "").strip()
    
    return row


In [11]:
test = "https://www.allfreecrochet.com/Crochet-Afghan-Patterns/Spring-Breeze-Baby-Blanket-187303"
scrape_product(test)

{'description': '"Wrap your little one in the soft, airy charm of the Spring Breeze Baby Blanket – Free Crochet Pattern that’s as refreshing as a warm spring morning! This delightful design features a soothing rhythm of two rows of cozy double crochet, followed by a playful eyelet row that adds just the right touch of texture. Then, just when you think you’ve got the pattern down, a second, slightly different eyelet row keeps things interesting and fun for your hook. The result is a light and breezy baby blanket with a beautiful balance of simplicity and whimsy which is perfect for springtime snuggles or thoughtful handmade gifts!"',
 'skill_level': 'Intermediate',
 'Crochet Hook': 'H/8 or 5 mm hook',
 'Yarn Weight': 'Finished Size',
 'Finished Size': 'Pin'}

In [13]:
base_url = "https://www.allfreecrochet.com"
rows = []

response = requests.get(base_url, headers={"User-Agent": "Mozilla/5.0"})
soup = BeautifulSoup(response.text, "html.parser")

cards = soup.select("div.articleCell")
if not cards:
    print("No more products found. Ending scrape.")

for card in cards:
    link = card.select_one("a[href]")
    product_link = base_url + link["href"] 

    img_tag = card.select_one("img")
    image_link = img_tag["src"] if img_tag else None

    name_tag = card.select_one("figcaption a")
    name = name_tag.get_text(strip=True) if name_tag else None

    product_data = scrape_product(product_link)

    product_data["name"] = name
    product_data["pattern_link"] = product_link
    product_data["image_link"] = image_link

    rows.append(product_data)
    time.sleep(1)

df = pd.DataFrame(rows)

In [14]:
df.head()

,description,Notes,skill_level,Crochet Hook,Yarn Weight,Crochet Gauge,Finished Size,name,pattern_link,image_link
0,Test your crochet skills with this amazing Rai...,Crochet Hook,Intermediate,M/13 or 9 mm hook,Crochet Gauge,Finished Size,Pin,Rainbow Tunisian Entrelac Crochet Baby Blanket...,https://www.allfreecrochet.com/Baby-Afghan-Cro...,//irepo.primecp.com/2015/07/227100/bernat-soft...
1,We’ve all been there: you’re digging through y...,NaN,NaN,NaN,NaN,NaN,NaN,47+ One Skein Crochet Patterns,https://www.allfreecrochet.com/Miscellaneous-C...,//irepo.primecp.com/2019/04/407909/AFC-One-Ske...
2,If you've been searching for easy afghan patte...,NaN,Beginner,H/8 or 5 mm hook,Crochet Gauge,Finished Size,Pin,World's Easiest Afghan,https://www.allfreecrochet.com/Crochet-Afghan-...,//irepo.primecp.com/2014/12/204013/Worlds-Easi...
3,"""We absolutely love long scarves. The type tha...",NaN,Easy,H/8 or 5 mm hook,Crochet Gauge,Finished Size,Pin,Spring Oversized Long Crochet Scarf,https://www.allfreecrochet.com/Scarves/Spring-...,//irepo.primecp.com/2025/07/602825/1752522717_...
4,"""Wrap your little one in the soft, airy charm ...",NaN,Intermediate,H/8 or 5 mm hook,Finished Size,NaN,Pin,Spring Breeze Baby Blanket,https://www.allfreecrochet.com/Crochet-Afghan-...,//irepo.primecp.com/2025/06/601674/1749590604_...
